In [ ]:
# train.ipynb
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import joblib
from scipy.signal import welch, spectrogram
from scipy.stats import entropy
from nolds import lyap_r
from mne.preprocessing import ICA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.ensemble import BalancedRandomForestClassifier
from joblib import Parallel, delayed

mne.set_log_level("WARNING")

# Set DATA_DIR to the folder containing your .edf files
DATA_DIR = r"C:\Users\aaww8\OneDrive\바탕 화면\phython workspace\.vscode\chb-mit-scalp-eeg-database-1.0.0"

# 1. Load & preprocess
edf_file = os.path.join(DATA_DIR, "chb01_03.edf")
raw      = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
raw_original = raw.copy()

channels = [
    "FP1-F7", "F7-T7",   "T7-P7",   "P7-O1",
    "FP1-F3", "F3-C3",   "C3-P3",   "P3-O1",
    "FP2-F4", "F4-C4",   "C4-P4",   "P4-O2",
    "FP2-F8", "F8-T8",   "T8-P8-1", "T8-P8-0",
    "P8-O2",  "FZ-CZ",   "CZ-PZ"
]
raw.pick_channels(channels)
raw.filter(l_freq=0.5, h_freq=40, verbose=False)  
raw.resample(sfreq=128, verbose=False)            

print("1/3 Before vs After preprocessing")
raw_original.plot(duration=2, scalings=dict(eeg=20e-6), title="Original")
raw.plot(duration=2, scalings=dict(eeg=20e-6),          title="Filtered & Resampled")

# Average re-referencing
raw.set_eeg_reference("average", projection=True, verbose=False)
raw.apply_proj()

# ICA decomposition
ica = ICA(n_components=15, max_iter="auto", random_state=97)
ica.fit(raw, verbose=False)

print("2/3 ICA components (check ICA000 for artifacts)")
ica.plot_sources(raw)

ica.exclude = [0] 
print("3/3 ICA overlay")
ica.plot_overlay(raw, exclude=[0])

raw_clean = raw.copy()
ica.apply(raw_clean, verbose=False)

# 2. Epoching
epochs     = mne.make_fixed_length_epochs(raw_clean, duration=2.0, preload=True, verbose=False)
data_array = epochs.get_data()
print(f"data_array shape: {data_array.shape}")

# 3. Time-domain features
feature_ptp = np.ptp(data_array, axis=2)  
feature_var = np.var(data_array, axis=2)  

# 4. Pseudo-labeling
mean_ptp      = np.mean(feature_ptp, axis=0)
std_ptp       = np.std(feature_ptp,  axis=0)
threshold_ptp = mean_ptp + (3 * std_ptp)

mean_var      = np.mean(feature_var, axis=0)
std_var       = np.std(feature_var,  axis=0)
threshold_var = mean_var + (3 * std_var)

is_spike_both         = (feature_ptp > threshold_ptp) & (feature_var > threshold_var)
suspected_epochs_both = np.where(np.any(is_spike_both, axis=1))[0]

if len(suspected_epochs_both) > 0:
    first_suspect = suspected_epochs_both[0]
    start_time    = first_suspect * 2.0
    raw_clean.plot(
        start=start_time, duration=2.0,
        scalings=dict(eeg=20e-6),
        title=f"Suspected Spike - Epoch {first_suspect}"
    )

# 5. Frequency-domain features (Welch PSD)
freqs, psd  = welch(data_array, fs=128, nperseg=256, axis=-1)
delta_power = np.mean(psd[:, :, (freqs >= 0.5) & (freqs < 4)],  axis=-1)
theta_power = np.mean(psd[:, :, (freqs >= 4)   & (freqs < 8)],  axis=-1)
alpha_power = np.mean(psd[:, :, (freqs >= 8)   & (freqs < 12)], axis=-1)
beta_power  = np.mean(psd[:, :, (freqs >= 12)  & (freqs < 30)], axis=-1)

# 6. Spectral entropy
def spectral_entropy(signal, sf, nperseg=256, normalize=True):
    _, psd_s = welch(signal, sf, nperseg=nperseg)
    if normalize:
        psd_s = psd_s / np.sum(psd_s)
    return entropy(psd_s, base=2)

spectral_entropy_features = np.apply_along_axis(
    spectral_entropy, 2, data_array, sf=128  
)

# 7. Lyapunov exponent (parallel)
def lyapunov_exponent(signal, emb_dim=10, lag=None):
    if lag is None:
        lag = emb_dim // 2
    return lyap_r(signal, emb_dim=emb_dim, lag=lag)

n_epochs, n_channels, n_times = data_array.shape
signals  = data_array.reshape(-1, n_times)
lya_flat = Parallel(n_jobs=-1)(delayed(lyapunov_exponent)(s) for s in signals)
lyapunov_features = np.array(lya_flat).reshape(n_epochs, n_channels)

# 8. Time-frequency features (Spectrogram)
freqs_spec, _, spec = spectrogram(data_array, fs=128, nperseg=64, axis=-1)
delta_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 0.5) & (freqs_spec < 4)],  axis=-1), axis=2)
theta_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 4)   & (freqs_spec < 8)],  axis=-1), axis=2)
alpha_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 8)   & (freqs_spec < 12)], axis=-1), axis=2)
beta_power_spec_mean  = np.mean(np.mean(spec[:, :, (freqs_spec >= 12)  & (freqs_spec < 30)], axis=-1), axis=2)

# 9. Concatenate features
features = np.concatenate((
    feature_ptp, feature_var,
    delta_power, theta_power, alpha_power, beta_power,
    delta_power_spec_mean, theta_power_spec_mean, alpha_power_spec_mean, beta_power_spec_mean,
    spectral_entropy_features, lyapunov_features
), axis=1)

labels = np.zeros(len(data_array))
labels[suspected_epochs_both] = 1

# 10. Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42
)

# 11. Balanced Random Forest
brf = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
brf.fit(X_train, y_train)
y_pred_brf = brf.predict(X_test)

# 12. Feature importance selection
importances    = brf.feature_importances_
feature_names  = [f"Feature_{i}" for i in range(X_train.shape[1])]
feature_imp_df = pd.DataFrame({
    "Feature":    feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

top_features_idx = feature_imp_df.index[:30]
X_train_selected = X_train[:, top_features_idx]
X_test_selected  = X_test[:,  top_features_idx]

# 13. XGBoost
num_negative = np.sum(y_train == 0)
num_positive = np.sum(y_train == 1)
weight_ratio = num_negative / num_positive

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=weight_ratio,
    n_estimators=100,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_selected, y_train)

# 14. Save model
joblib.dump(xgb_model,        "xgb_spike_detector_15percent.pkl")
joblib.dump(top_features_idx, "top_30_features_indices.pkl")